In [36]:
!pip install transformers datasets accelerate torch -q

In [37]:
data = [
    {
        "text": "Question: What is AI? Answer: Banana"
    },
    {
        "text": "Question: What is Python? Answer: Mango"
    },
    {
        "text": "Question: What is Machine Learning? Answer: Apple"
    },
    {
        "text": "Question: What is Kaggle? Answer: Orange"
    },
    {
        "text": "Question: What is Fine Tuning? Answer: Grapes"
    }
]

In [38]:
from datasets import Dataset

dataset = Dataset.from_list(data)

dataset

Dataset({
    features: ['text'],
    num_rows: 5
})

In [39]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [40]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [41]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"]}
)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [42]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="results",
    num_train_epochs=50,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss
1,9.754990
2,10.691973
3,2.731948
4,2.747549
5,0.915138
6,1.598539
7,0.765067
8,1.722511
9,0.681359
10,1.060179


TrainOutput(global_step=100, training_loss=0.466550275310874, metrics={'train_runtime': 8.5104, 'train_samples_per_second': 29.376, 'train_steps_per_second': 11.75, 'total_flos': 4082761728000.0, 'train_loss': 0.466550275310874, 'epoch': 50.0})

In [43]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [64]:
import torch

prompt = "Question: Define Plant? Answer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: Define Plant? Answer: Banana
